# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Chặng 1: đếm ảnh/box, kiểm tra không có session nào nằm ở 2 split, vẽ 6 ảnh GT
from scripts.labcommon import read_jsonl
sess = {}
for s in ("train", "val", "test"):
    rows = read_jsonl(f"data/splits/{s}.jsonl")
    sess[s] = {r["session"] for r in rows}
    print(s, len(rows), "ảnh,", len(sess[s]), "session")
assert not (sess["train"] & sess["val"] or sess["train"] & sess["test"] or sess["val"] & sess["test"])
!python -m lab.viz gt --out report/figures/gt_samples.jpg

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Chặng 2: xem lab/labels.py (yolo_to_box, objects_to_answer, PROMPT_NORM1000)
from lab.labels import objects_to_answer, PROMPT_NORM1000
e = read_jsonl("data/splits/train.jsonl")[0]
print(PROMPT_NORM1000)
print(objects_to_answer(e["objects"]))
!python -m pytest lab/tests -q

## 3. E0: zero-shot

In [ ]:
# E0: lab/vlm.py::load_base (NF4, skip ViT+projector+lm_head) + lab/run.py::predict
# Đã chạy trên Kaggle T4: python kaggle/push.py e0 ; python kaggle/fetch.py e0
!python -m lab.run e0 --data data --out work/e0 && cp work/e0/e0.jsonl work/e0.jsonl

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Chặng 3: lab/vlm.py::encode_example (labels=-100 trên ảnh+prompt) và collate (cat pixel_values)
# Smoke test in: 220 token ảnh, decode(labels!=-100) == JSON + <|im_end|>, 252/348 module LoRA
!python -m lab.run smoke --data data --out work/smoke

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Chặng 4: lab/vlm.py::apply_setup — LoRA r=16 trên 252 module LLM (E1),
# + projector fp32 huấn luyện đầy đủ (E2), + LoRA 96 module ViT (E3)
from lab.vlm import LLM_TARGETS, VIT_TARGETS
print(LLM_TARGETS); print(VIT_TARGETS)

## 6. Vòng huấn luyện

In [ ]:
# lab/run.py::train — AdamW 2e-4, warmup 5% + cosine, bs 2 × accum 4, 300 bước, fp16 + GradScaler,
# gradient checkpointing; lưu adapter + projector.pt (E2/E3) mỗi 100 bước; run.json có VRAM/thời gian
for s in ("E1", "E2", "E3"):
    !python -m lab.run train --setup {s} --name {s.lower()} --data data --out work/{s.lower()}

## 7. Suy luận và chấm E1–E3

In [ ]:
# lab/run.py::cmd_train tự nạp lại model gốc -> adapter -> projector rồi suy luận test.
# Chấm (kèm --meta từ run.json): python kaggle/fetch.py e1  (tương tự e2, e3)
!python -m lab.viz table

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 (data_size): python kaggle/push.py e4_data_0.1 -- --setup <tốt nhất> --train-frac 0.1
# E5 (GRPO, reward IoU trong lab/reward.py): python kaggle/push.py e5 --from lab10-<tốt nhất> -- --impl trl
!python -m lab.grpo --help